In [3]:
import glob
import os
import pandas as pd # Przydatne, jeśli chcesz to zobaczyć w formie tabeli


In [4]:
RAW_DATA_ROOT = 'SoundData/RawData'
SUPPORTED_AUDIO_EXTENSIONS = ('*.wav', '*.flac', '*.mp3', '*.ogg') # Możesz dodać inne, jeśli potrzebujesz

# --- Generowanie listy plików ---
raw_audio_files = []
print(f"Skanowanie katalogu: {RAW_DATA_ROOT} w poszukiwaniu plików audio...")

for ext in SUPPORTED_AUDIO_EXTENSIONS:
    search_pattern = os.path.join(RAW_DATA_ROOT, '**', ext)
    found_files = glob.glob(search_pattern, recursive=True)
    raw_audio_files.extend(found_files)

len(raw_audio_files)

Skanowanie katalogu: SoundData/RawData w poszukiwaniu plików audio...


71

In [5]:
import os
import librosa
import warnings

def get_audio_info_in_folder(folder_path):
    """
    Zwraca liczbę nagrań i ich łączny czas trwania w danym folderze.

    Args:
        folder_path (str): Ścieżka do folderu zawierającego pliki audio.

    Returns:
        tuple: (total_files_found, total_duration_seconds)
               Zwraca (0, 0.0) jeśli folder nie istnieje lub nie zawiera plików audio.
    """
    if not os.path.isdir(folder_path):
        print(f"Błąd: Folder '{folder_path}' nie istnieje.")
        return 0, 0.0

    total_files_found = 0
    total_duration_seconds = 0.0
    supported_extensions = ['.wav', '.mp3', '.flac', '.ogg'] # Możesz dodać więcej, jeśli potrzebujesz

    print(f"Skanowanie folderu: {folder_path}")

    # Wyłącz ostrzeżenia librosa, które mogą być irytujące dla każdego pliku
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        for root, _, files in os.walk(folder_path):
            for file in files:
                # Sprawdź, czy plik ma obsługiwane rozszerzenie
                if any(file.lower().endswith(ext) for ext in supported_extensions):
                    file_path = os.path.join(root, file)
                    try:
                        # Wczytaj tylko nagłówek pliku, aby szybko uzyskać czas trwania
                        duration = librosa.get_duration(path=file_path)
                        total_duration_seconds += duration
                        total_files_found += 1
                    except Exception as e:
                        print(f"Nie udało się wczytać pliku '{file_path}'. Błąd: {e}")
                        continue

    # Przekształć łączny czas trwania na bardziej czytelny format
    minutes = int(total_duration_seconds // 60)
    seconds = total_duration_seconds % 60

    print(f"\nZnaleziono {total_files_found} nagrań.")
    print(f"Łączny czas trwania: {minutes} minut i {seconds:.2f} sekund.")

    return total_files_found, total_duration_seconds

In [7]:
path = r"C:\Users\Krysia\Desktop\Other\Projekty\Machine Learning\BoarSoundClassifier\Model\SoundData\RawData\BBC\City"
num_files, total_time = get_audio_info_in_folder(path)
print(num_files, total_time)

Skanowanie folderu: C:\Users\Krysia\Desktop\Other\Projekty\Machine Learning\BoarSoundClassifier\Model\SoundData\RawData\BBC\City

Znaleziono 19 nagrań.
Łączny czas trwania: 33 minut i 35.19 sekund.
19 2015.1877324263037


In [18]:
# Słowa kluczowe, których obecność w ścieżce lub nazwie pliku wyklucza plik
EXCLUDE_KEYWORDS = ['xeno-canto', 'Cornell']
raw_audio_files = sorted(list(set(raw_audio_files)))

print(f"Znaleziono wstępnie {len(raw_audio_files)} plików audio.")

filtered_audio_files = []
excluded_count = 0

for filepath in raw_audio_files:
    if any(keyword.lower() in filepath.lower() for keyword in EXCLUDE_KEYWORDS):
        excluded_count += 1
    else:
        filtered_audio_files.append(filepath)

raw_audio_files = filtered_audio_files
print(f"Wykluczono {excluded_count} plików zawierających słowa kluczowe: {EXCLUDE_KEYWORDS}.")
print(f"Pozostało {len(raw_audio_files)} plików audio do dalszego przetwarzania.")


Znaleziono wstępnie 71 plików audio.
Wykluczono 11 plików zawierających słowa kluczowe: ['xeno-canto', 'Cornell'].
Pozostało 60 plików audio do dalszego przetwarzania.


In [19]:
import pandas as pd
import librosa
import os

RAW_DATA_ROOT = 'SoundData/RawData'
SUPPORTED_AUDIO_EXTENSIONS = ('.wav', '.flac', '.mp3', '.ogg')
EXCLUDE_KEYWORDS = ['xeno-canto', 'Cornell'] #'BBC', 'Fesliyanstudios', 'Freesound'

print(f"Skanowanie katalogu: {RAW_DATA_ROOT} w poszukiwaniu plików audio...")

original_records_metadata = []

for root, _, files in os.walk(RAW_DATA_ROOT):
    for file in files:
        if file.lower().endswith(SUPPORTED_AUDIO_EXTENSIONS):
            full_filepath = os.path.join(root, file)
            if not any(keyword.lower() in full_filepath.lower() for keyword in EXCLUDE_KEYWORDS):
                try:
                    duration = librosa.get_duration(path=full_filepath)
                    adjusted_duration = max(duration, 4.0)
                    file_name_without_extension = os.path.splitext(file)[0]

                    original_records_metadata.append({
                        'original_recording_name': file_name_without_extension,
                        'original_recording_filepath': full_filepath,
                        'original_recording_duration_seconds': adjusted_duration, # Użyj zmienionej wartości
                    })
                except Exception as e:
                    print(f"Nie można przetworzyć pliku {full_filepath}: {e}")

original_records_df = []
if not original_records_metadata:
    print("Ostrzeżenie: Nie znaleziono żadnych plików audio spełniających kryteria.")
else:
    original_records_df = pd.DataFrame(original_records_metadata)

print(f"\nZnaleziono {len(original_records_df)} oryginalnych nagrań.")
print(f"Całkowity czas trwania: {original_records_df['original_recording_duration_seconds'].sum():.0f} sekund.")

# --- 2. Zapis Metadanych do Pliku CSV ---
original_records_metadata_file = 'original_records_metadata.csv'
original_records_df.to_csv(original_records_metadata_file, index=False)
print(f"\nMetadane oryginalnych nagrań (bez etykiet) zapisano do pliku: {original_records_metadata_file}")

Skanowanie katalogu: SoundData/RawData w poszukiwaniu plików audio...

Znaleziono 60 oryginalnych nagrań.
Całkowity czas trwania: 5400 sekund.

Metadane oryginalnych nagrań (bez etykiet) zapisano do pliku: original_records_metadata.csv


In [20]:
import pandas as pd

# --- Konfiguracja ---
METADATA_FILE = 'original_records_metadata.csv'
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_STATE = 42

tolerance = 1e-9
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < tolerance, "Suma proporcji musi być bardzo bliska 1.0"


# --- 1. Wczytanie metadanych oryginalnych nagrań ---
try:
    original_records_df = pd.read_csv(METADATA_FILE)
    print(f"Pomyślnie wczytano metadane z pliku: {METADATA_FILE}")
    print(f"Liczba nagrań w metadanych: {len(original_records_df)}")
    print(f"Całkowity czas trwania wszystkich oryginalnych nagrań: {original_records_df['original_recording_duration_seconds'].sum():.0f} sekund.")

except FileNotFoundError:
    print(f"Błąd: Plik '{METADATA_FILE}' nie został znaleziony.")
    print("Upewnij się, że uruchomiłeś/aś poprzednią komórkę, która generuje ten plik.")

    # Tworzenie DUMMY DANYCH do podziału, jeśli plik nie istnieje (tylko do demonstracji!)
    print("\nGeneruję przykładowe metadane dla demonstracji podziału danych!")
    data_dummy = {
        'original_recording_name': ['orig_A', 'orig_B', 'orig_C', 'orig_D', 'orig_E', 'orig_F', 'orig_G', 'orig_H', 'orig_I', 'orig_J'],
        'original_recording_filepath': [f'dummy_path/{f}.wav' for f in ['orig_A', 'orig_B', 'orig_C', 'orig_D', 'orig_E', 'orig_F', 'orig_G', 'orig_H', 'orig_I', 'orig_J']],
        'original_recording_duration_seconds': [10, 15, 8, 20, 12, 17, 9, 22, 11, 14],
    }
    original_records_df = pd.DataFrame(data_dummy)
    print(f"Liczba DUMMY nagrań: {len(original_records_df)}")


# --- 2. Przygotowanie danych do podziału ---
# W tej fazie nie mamy etykiet, więc podział będzie bazował tylko na czasie trwania
# Chcemy, aby suma czasów trwania w każdym zbiorze odpowiadała proporcjom.
# train_test_split domyślnie dzieli losowo wiersze, co dla czasów trwania jest w porządku,
# ponieważ dążymy do równomiernego rozłożenia łącznego czasu.

# Dane do podziału: nazwy plików i ich czasy trwania
data_to_split = original_records_df[['original_recording_name', 'original_recording_duration_seconds']]

# Własna funkcja do podziału z uwzględnieniem czasu trwania
def split_by_duration(df, train_ratio, val_ratio, test_ratio, random_state):
    """
    Dzieli DataFrame na zbiory treningowy, walidacyjny i testowy
    starając się zbalansować łączny czas trwania.
    """
    total_duration = df['original_recording_duration_seconds'].sum()

    # Przeliczanie docelowych czasów trwania
    target_train_duration = total_duration * train_ratio
    target_val_duration = total_duration * val_ratio
    target_test_duration = total_duration * test_ratio

    # Losowa kolejność plików, by zapewnić losowość w wyborze
    shuffled_df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    train_files = []
    val_files = []
    test_files = []

    current_train_duration = 0
    current_val_duration = 0
    current_test_duration = 0

    # Rozdzielanie plików iteracyjnie, próbując zbliżyć się do docelowych czasów
    for index, row in shuffled_df.iterrows():
        name = row['original_recording_name']
        duration = row['original_recording_duration_seconds']

        # Prosta heurystyka: najpierw próbujemy dodać do zbioru, który najbardziej "potrzebuje" czasu
        # To nie gwarantuje idealnego rozkładu, ale jest prostym podejściem.
        # Bardziej zaawansowane algorytmy wymagają optymalizacji lub specjalizowanych bibliotek.

        if current_train_duration + duration <= target_train_duration or \
           (current_train_duration / target_train_duration < current_val_duration / target_val_duration and \
            current_train_duration / target_train_duration < current_test_duration / target_test_duration):

            train_files.append(name)
            current_train_duration += duration
        elif current_val_duration + duration <= target_val_duration or \
             (current_val_duration / target_val_duration < current_test_duration / target_test_duration):

            val_files.append(name)
            current_val_duration += duration
        else:
            test_files.append(name)
            current_test_duration += duration

    return train_files, val_files, test_files, current_train_duration, current_val_duration, current_test_duration

train_names, val_names, test_names, train_dur, val_dur, test_dur = \
    split_by_duration(original_records_df, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, RANDOM_STATE)

print("\n--- Podsumowanie Podziału Oryginalnych Nagrań ---")
print(f"Trening: {len(train_names)} plików ({train_dur:.0f} sek, {train_dur/original_records_df['original_recording_duration_seconds'].sum():.2%})")
print(f"Walidacja: {len(val_names)} plików ({val_dur:.0f} sek, {val_dur/original_records_df['original_recording_duration_seconds'].sum():.2%})")
print(f"Test: {len(test_names)} plików ({test_dur:.0f} sek, {test_dur/original_records_df['original_recording_duration_seconds'].sum():.2%})")

# --- 3. Zapisanie list nazw plików do CSV ---
pd.DataFrame({'original_recording_name': train_names}).to_csv('train_original_names.csv', index=False)
pd.DataFrame({'original_recording_name': val_names}).to_csv('val_original_names.csv', index=False)
pd.DataFrame({'original_recording_name': test_names}).to_csv('test_original_names.csv', index=False)

print("\nListy nazw oryginalnych nagrań dla każdego zbioru (bez rozszerzeń) zapisane do plików CSV:")
print(" - train_original_names.csv")
print(" - val_original_names.csv")
print(" - test_original_names.csv")

print("\nPrzykładowe nazwy plików dla zbioru treningowego:")
print(train_names[:10]) # Wyświetl pierwsze 10 nazw dla podglądu

Pomyślnie wczytano metadane z pliku: original_records_metadata.csv
Liczba nagrań w metadanych: 60
Całkowity czas trwania wszystkich oryginalnych nagrań: 5400 sekund.

--- Podsumowanie Podziału Oryginalnych Nagrań ---
Trening: 42 plików (3779 sek, 69.98%)
Walidacja: 14 plików (786 sek, 14.55%)
Test: 4 plików (835 sek, 15.47%)

Listy nazw oryginalnych nagrań dla każdego zbioru (bez rozszerzeń) zapisane do plików CSV:
 - train_original_names.csv
 - val_original_names.csv
 - test_original_names.csv

Przykładowe nazwy plików dla zbioru treningowego:
['NHU05040109', 'NHU05040114', '07037426', 'NHU05101029', 'NHU05095004', '425241__garuda1982__wild-boar-squeak-field-recording-with-tascam-dr-70d-rode-ntg4', '07022393', 'Pig-Heavy-Screaming-A', 'NHU05095003', '57027__noisecollector__deepwoods']


In [21]:
#TODO porównajmy jeszcze jak to wygląda po faktycznym podziale